
# Steam Bronze Layer Ingestion

## Objective

This notebook ingests the raw Steam dataset into the Bronze layer.

Responsibilities:
- ingest raw JSON data from S3
- preserve raw structure
- apply minimal normalization
- add ingestion metadata
- persist raw Delta table

Target table:
- `steam.bronze.steam_games_bronze`

Bronze principles:
- preserve lineage
- avoid business transformations
- keep raw nested schema
- ensure reproducible ingestion


## Imports

In [0]:

from pyspark.sql import functions as F



# 1. Source Configuration

Objectives:
- define raw source path
- centralize ingestion configuration


In [0]:

SOURCE_PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

BRONZE_TABLE = "steam.bronze.steam_games_bronze"



# 2. Raw Data Ingestion

Objectives:
- ingest raw JSON dataset
- preserve nested schema
- validate ingestion completeness


In [0]:

steam_df = (
    spark.read
    .option("multiline", "true")
    .json(SOURCE_PATH)
)


In [0]:

display(
    spark.createDataFrame([
        ("rows", steam_df.count()),
        ("columns", len(steam_df.columns))
    ], ["metric", "value"])
)


metric,value
rows,55691
columns,2


In [0]:

steam_df.printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-


# 3. Bronze Layer Minimal Cleaning

Bronze transformations must remain minimal.

Applied rules:
- remove duplicate rows
- add ingestion timestamp
- preserve raw nested schema


In [0]:

df_bronze = (

    steam_df

    .dropDuplicates()

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)



# 4. Bronze Validation

Objectives:
- validate ingestion integrity
- ensure dataset consistency


In [0]:

validation_summary = {

    "duplicate_rows_removed":
        steam_df.count() - df_bronze.count(),

    "final_row_count":
        df_bronze.count(),

    "null_data_rows":
        df_bronze.filter(
            F.col("data").isNull()
        ).count()
}

validation_summary


{'duplicate_rows_removed': 0, 'final_row_count': 55691, 'null_data_rows': 0}


# 5. Catalog & Schema Initialization

Objectives:
- create Bronze catalog hierarchy
- prepare managed Delta table storage


In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS steam.bronze")


DataFrame[]


# 6. Persist Bronze Table

Objectives:
- persist managed Delta table
- preserve schema evolution compatibility


In [0]:

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)



# 7. Post-Ingestion Validation

Objectives:
- validate persisted table
- inspect final schema


In [0]:

df_check = spark.table(BRONZE_TABLE)

display(
    spark.createDataFrame([
        ("rows", df_check.count()),
        ("columns", len(df_check.columns))
    ], ["metric", "value"])
)


metric,value
rows,55691
columns,3


In [0]:

df_check.printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-